# 🍳 Building a Recipe Assistant MCP Server on Union

## Help AI Agents Find Delicious Recipes!

This tutorial walks you through building a **Model Context Protocol (MCP)** server that allows AI agents to search for recipes. You'll create a recipe assistant that can find meals by ingredients, dietary needs, and nutritional goals using the [Spoonacular Food API](https://spoonacular.com/food-api).

### What You'll Learn

1. **What MCP is** and why it's useful for AI agents
2. **Setting up the Spoonacular API** (just one API key!)
3. **Building MCP tools** for recipe search
4. **Testing locally** before deployment
5. **Deploying to Union** for scalable, managed execution

---

## Part 1: Understanding MCP

### What is the Model Context Protocol?

The **Model Context Protocol (MCP)** is an open standard developed by Anthropic that enables AI assistants to securely connect with external data sources and tools. Think of it as a universal adapter that allows AI agents to interact with APIs.

```
┌─────────────────┐                          ┌─────────────────┐                    ┌─────────────────┐
│                 │      MCP Protocol        │                 │      API Calls     │                 │
│    AI Agent     │◄────────────────────────►│   MCP Server    │◄──────────────────►│   Spoonacular   │
│  (Claude, etc.) │   - list tools           │  (Your Server)  │                    │    Food API     │
│                 │   - call tools           │                 │                    │                 │
└─────────────────┘                          └─────────────────┘                    └─────────────────┘
```

### Why a Recipe Assistant?

Everyone eats! A recipe assistant is:
- **Universally relatable** - helps with daily meal planning
- **Practical** - real value for users
- **Great for demos** - visual, engaging results

---

## Part 2: Get Your API Key (2 minutes!)

### Setting Up Spoonacular

1. Go to [spoonacular.com/food-api](https://spoonacular.com/food-api)
2. Click **Start Now** and create a free account
3. Copy your API key from the dashboard

**That's it!** The free tier includes 150 points/day - plenty for this tutorial.

### API Key Safety

Never commit your API key to git! We'll use environment variables.

In [ ]:
# Set your API key here (or use a .env file)
import os

# Option 1: Set directly (for testing only - don't commit this!)
# os.environ["SPOONACULAR_API_KEY"] = "your-api-key-here"

# Option 2: Load from .env file (recommended)
from dotenv import load_dotenv
load_dotenv()

# Check if API key is set
api_key = os.getenv("SPOONACULAR_API_KEY")
if api_key:
    print(f"✅ API key loaded! (ends with ...{api_key[-4:]})")
else:
    print("❌ No API key found. Set SPOONACULAR_API_KEY environment variable.")

---

## Part 3: Building the Recipe Client

Let's create a simple client to interact with the Spoonacular API. This will be the foundation for our MCP tools.

In [ ]:
import httpx

class RecipeClient:
    """Simple client for Spoonacular Food API."""
    
    BASE_URL = "https://api.spoonacular.com"
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.client = httpx.Client(timeout=30.0)
    
    def _request(self, endpoint: str, params: dict = None) -> dict:
        """Make API request."""
        params = params or {}
        params["apiKey"] = self.api_key
        response = self.client.get(f"{self.BASE_URL}{endpoint}", params=params)
        response.raise_for_status()
        return response.json()
    
    def search_recipes(self, query: str, number: int = 5) -> list:
        """Search for recipes by name/description."""
        result = self._request("/recipes/complexSearch", {"query": query, "number": number})
        return result.get("results", [])
    
    def search_by_ingredients(self, ingredients: list[str], number: int = 5) -> list:
        """Find recipes using ingredients you have."""
        return self._request("/recipes/findByIngredients", {
            "ingredients": ",".join(ingredients),
            "number": number,
            "ranking": 1,  # Maximize used ingredients
        })
    
    def get_recipe_info(self, recipe_id: int) -> dict:
        """Get detailed recipe information."""
        return self._request(f"/recipes/{recipe_id}/information")

# Create client
client = RecipeClient(api_key)
print("✅ Recipe client ready!")

In [ ]:
# 🧪 Test: Search for pasta recipes
recipes = client.search_recipes("pasta carbonara", number=3)

print("🍝 Found recipes:")
for r in recipes:
    print(f"  - {r['title']} (ID: {r['id']})")

In [ ]:
# 🧪 Test: "What's in my fridge?" search
my_ingredients = ["chicken", "rice", "garlic", "onion"]
recipes = client.search_by_ingredients(my_ingredients, number=3)

print(f"🍗 Recipes using {my_ingredients}:")
for r in recipes:
    used = [i["name"] for i in r.get("usedIngredients", [])]
    missing = [i["name"] for i in r.get("missedIngredients", [])]
    print(f"\n  📖 {r['title']}")
    print(f"     ✅ Uses: {', '.join(used)}")
    print(f"     🛒 Need: {', '.join(missing) if missing else 'Nothing else!'}")

---

## Part 4: Creating MCP Tools

Now let's wrap our recipe functionality into MCP tools using `fastmcp`.

In [ ]:
from fastmcp import FastMCP

# Initialize the MCP server
mcp = FastMCP(
    name="recipe-assistant",
    instructions="""
    You are a helpful recipe assistant. You can:
    - Search for recipes by name or description
    - Find recipes using ingredients the user has
    - Get detailed recipe information
    
    Always be helpful and suggest alternatives when appropriate!
    """
)

print("✅ MCP server initialized!")

In [ ]:
# Define MCP tools using decorators

@mcp.tool()
async def search_recipes(
    query: str,
    cuisine: str = None,
    diet: str = None,
    number: int = 5,
) -> list[dict]:
    """
    Search for recipes by name, cuisine, or dietary preference.
    
    Args:
        query: What to search for (e.g., "pasta", "quick dinner", "chocolate cake")
        cuisine: Optional cuisine filter (italian, mexican, asian, etc.)
        diet: Optional diet filter (vegan, vegetarian, gluten free, keto, etc.)
        number: How many recipes to return (default 5)
    
    Returns:
        List of matching recipes with titles and IDs
    """
    params = {"query": query, "number": number}
    if cuisine:
        params["cuisine"] = cuisine
    if diet:
        params["diet"] = diet
    
    result = client._request("/recipes/complexSearch", params)
    return result.get("results", [])


@mcp.tool()
async def search_by_ingredients(
    ingredients: list[str],
    number: int = 5,
) -> list[dict]:
    """
    Find recipes using ingredients you have on hand.
    
    This is the "what's in my fridge" tool - perfect for reducing food waste!
    
    Args:
        ingredients: List of ingredients you have (e.g., ["chicken", "rice", "garlic"])
        number: How many recipes to return (default 5)
    
    Returns:
        Recipes showing which ingredients are used and which you'd need to buy
    """
    recipes = client.search_by_ingredients(ingredients, number)
    
    # Format for clarity
    return [
        {
            "id": r["id"],
            "title": r["title"],
            "used_ingredients": [i["name"] for i in r.get("usedIngredients", [])],
            "missing_ingredients": [i["name"] for i in r.get("missedIngredients", [])],
        }
        for r in recipes
    ]


@mcp.tool()
async def get_recipe_details(recipe_id: int) -> dict:
    """
    Get full details for a specific recipe including ingredients and instructions.
    
    Args:
        recipe_id: The recipe ID (from a previous search)
    
    Returns:
        Complete recipe with ingredients, instructions, and nutrition info
    """
    recipe = client.get_recipe_info(recipe_id)
    
    return {
        "title": recipe.get("title"),
        "servings": recipe.get("servings"),
        "ready_in_minutes": recipe.get("readyInMinutes"),
        "ingredients": [
            f"{i['amount']} {i['unit']} {i['name']}"
            for i in recipe.get("extendedIngredients", [])
        ],
        "instructions": recipe.get("instructions", "No instructions available"),
        "diets": recipe.get("diets", []),
    }


print("✅ MCP tools defined!")
print("\nRegistered tools:")
print("  - search_recipes")
print("  - search_by_ingredients") 
print("  - get_recipe_details")

---

## Part 5: Testing the MCP Tools

Let's test our tools before deploying!

In [ ]:
# Test: Search for vegan Italian recipes
print("🧪 Testing search_recipes...")
results = await search_recipes("pasta", cuisine="italian", diet="vegan", number=3)
for r in results:
    print(f"  🍝 {r['title']}")

In [ ]:
# Test: What can I make with these ingredients?
print("\n🧪 Testing search_by_ingredients...")
results = await search_by_ingredients(["salmon", "lemon", "dill"], number=3)
for r in results:
    print(f"\n  🐟 {r['title']}")
    print(f"     ✅ Uses: {', '.join(r['used_ingredients'])}")
    print(f"     🛒 Need: {', '.join(r['missing_ingredients']) or 'Nothing!'}")

In [ ]:
# Test: Get full recipe details
print("\n🧪 Testing get_recipe_details...")
if results:
    recipe_id = results[0]["id"]
    details = await get_recipe_details(recipe_id)
    
    print(f"\n📖 {details['title']}")
    print(f"   ⏱️  Ready in {details['ready_in_minutes']} minutes")
    print(f"   🍽️  Serves {details['servings']}")
    print(f"\n   Ingredients:")
    for ing in details['ingredients'][:5]:  # First 5
        print(f"     • {ing}")
    if len(details['ingredients']) > 5:
        print(f"     ... and {len(details['ingredients']) - 5} more")

---

## Part 6: Running the MCP Server

To run locally: `python server.py`

To connect to Cursor, add to `~/.cursor/mcp.json`:

```json
{
  "mcpServers": {
    "recipe-assistant": {
      "command": "python",
      "args": ["server.py"],
      "cwd": "/path/to/tutorials/mcp",
      "env": {"SPOONACULAR_API_KEY": "your-api-key"}
    }
  }
}
```

Then try: "What can I make with chicken and rice?"

---

## Part 7: Deploying to Union

Deploy your MCP server to Union for production use!

### 1. Store your API key as a Union secret

```bash
union create secret SPOONACULAR_API_KEY
```

### 2. Build and deploy

```bash
python deploy.py --build   # Build container
python deploy.py           # Run test
```

### Benefits of Union deployment:
- **Managed infrastructure** - no servers to maintain
- **Secure secrets** - API keys stored safely
- **Scalability** - handle multiple requests
- **Monitoring** - track all tool calls

---

## 🎉 Summary

You built a Recipe Assistant MCP server that can:

✅ Search recipes by name, cuisine, and dietary needs  
✅ Find recipes using ingredients you have ("what's in my fridge")  
✅ Get detailed recipe information with ingredients and instructions  

### Next Steps

- Add more tools: nutrition search, meal planning, wine pairing
- Build a weekly meal prep workflow
- Integrate with shopping list APIs

### Resources

- [Spoonacular API Docs](https://spoonacular.com/food-api/docs)
- [Union MCP Repository](https://github.com/unionai-oss/union-mcp)
- [Model Context Protocol](https://modelcontextprotocol.io/)

Happy cooking! 🍳